# 🔑 SSH: Keys & Remote Servers

**DigiMaster Pre-Master Days 2026**

Today you get a digital identity. You will create your own **SSH key**, connect it to **GitHub** — which unlocks the `git push` we postponed in Notebook 2 — and see exactly how that same key gets you into a **remote server**.

| | |
|---|---|
| ⏱️ **Time needed** | about 45–60 minutes (some of it reading — see the box below) |
| 🧠 **You need to know** | `01-command-line` (terminal basics); `02-git` (for the GitHub part) |
| 🛠️ **You will install** | nothing — SSH is already on your computer |
| 🎓 **Done together in class** | sections 5, 6, 7 and 9 — they need the course server, which does not exist yet. Read them now, we do them in the lecture |
| ➡️ **Next** | `04-docker` |

> 🎓 **One thing is different in this notebook — please read this first.**
>
> Sections **5, 6, 7 and 9** need a **remote server to log into**, and **there is no course server running yet.** So for now those four sections are **for reading only** — read them, but do not expect to type along.
>
> **We will do them together in the lecture, live, on the real server.** You will get your username and the server address then, and we walk through logging in, `ssh-copy-id`, the address book and `scp` as a group. Nothing is lost by waiting: reading them now means you will recognize every step when we get there.
>
> **What to do on your own, now:** sections **1–4** — understand SSH, check it is installed, create your key pair — and section **8**, connecting your key to GitHub. Section 8 works today and is the one that unlocks `git push`. Sections 10–12 are worth reading either way.

> 📖 **How to use this notebook** — as always: read here, type the gray boxes into **your own terminal**, press Enter, compare with the expected output. Good news: SSH works **identically on every system** — 🪟 Windows users type everything in their Ubuntu terminal from Notebook 1, exactly like the 🐧 Linux users.

## 1. What is SSH, and why does everyone use it?

**SSH** stands for **S**ecure **SH**ell. It lets you **open a terminal on another computer, across the internet** — securely.

Here is the mental picture. In Notebook 1 you typed `ls` and *your laptop* answered. With SSH, you type:

```
ssh anna@server.university.edu
```

…and from that moment, everything you type runs **on the university's server**, and the answers come back to your screen. Same window, different computer. Like remote-controlling a machine — except with text instead of screen-sharing.

Why this matters for you:

- **The real machines are remote.** Company data pipelines, websites, and analytics don't run on laptops — they run on servers in data centers. Those servers have no screen, no mouse, no buttons. SSH **is** the way in.
- **"Secure" is not decoration.** Everything you send is encrypted — passwords, commands, data. This is why SSH (from 1995!) is still the standard today.
- **You will use it in this course** — to prove your identity to GitHub (section 8, today), and to log into the course server (which we do together in the lecture).

## 2. Passwords vs. keys — the padlock idea 🔓

You *can* use SSH with a password. But professionals use **key pairs**, and once you see why, you will too:

A key pair is **two files that belong together**, created in one go on your laptop:

| File | Called | Think of it as | Rules |
|---|---|---|---|
| `id_ed25519.pub` | **public key** | a **padlock** 🔓 | Copy it freely — hand it to any server, paste it into GitHub. Public = public. |
| `id_ed25519` | **private key** | the **only key** 🗝️ that opens those padlocks | **Stays on your laptop. Forever.** Never email it, never paste it anywhere, never show it to anyone. |

How logging in works after setup:

1. You hang your **padlock** (public key) on a server — one time.
2. When you connect, the server rattles the padlock: *"prove you have the matching key!"*
3. Your laptop proves it **without sending the key anywhere** (cryptographic magic).
4. Door opens. No password typed, nothing secret traveled over the network.

Why this beats passwords:

- **Nothing to intercept or guess** — the private key never leaves your machine, and it is astronomically stronger than any password you could memorize.
- **One identity everywhere** — the same single key pair works for the course server, GitHub, and every machine of your future career. Hang copies of your padlock everywhere; keep the one key.
- **Convenient** — after setup, `ssh` just… logs in. No typing passwords all day.

## 3. Check that SSH is on your computer

SSH ships with every modern system — let's verify. Type:

```
ssh -V
```

(capital V!) Expected — one line naming a version, something like:

```
OpenSSH_9.6p1, LibreSSL 3.3.6
```

- 🍎 **macOS** — always there. ✅
- 🐧 **Linux** — always there. ✅ (In the rare case it's missing: `sudo apt install openssh-client`.)
- 🪟 **Windows** — your Ubuntu terminal ships the complete SSH toolbox, exactly like any Linux. ✅ (Notebook 1's setup, paying off again.)

> ✅ **Checkpoint** — `ssh -V` prints a version line. Any version is fine.

## 4. Create your key pair 🗝️

One command, on **all** systems (put **your** email in the quotes — it's just a label so the key can be recognized as yours):

```
ssh-keygen -t ed25519 -C "anna.example@gmail.com"
```

Decoded: `ssh-keygen` = "make me a key pair" · `-t ed25519` = of the modern type Ed25519 · `-C "..."` = with this comment/label attached.

The program now asks you **three questions**. Here is exactly how to answer:

### Question 1 — where to save the key?

```
Enter file in which to save the key (/Users/anna/.ssh/id_ed25519):
```

**Just press Enter.** The suggested place (a hidden folder `.ssh` in your home folder) is exactly where every program will look for it. (On a brand-new machine it also prints `Created directory '…/.ssh'` — that is ssh-keygen making the hidden folder for you. Good.)

> ⚠️ **Careful** — if it instead says `…/id_ed25519 already exists. Overwrite (y/n)?` — **type `n`!** You already have a key (perhaps from an earlier course). Don't destroy it; skip to **"4½ — Only if the key already existed"** below.

### Questions 2 & 3 — a passphrase?

```
Enter passphrase (empty for no passphrase):
Enter same passphrase again:
```

A **passphrase** is a password *for the key file itself*: if someone steals your laptop, they can't use your key without it. The trade-off: you type it when the key is used.

- **Simplest for this course: press Enter twice** (no passphrase). Perfectly common for student use.
- More security-conscious: type a passphrase you will remember (typing shows nothing on screen — normal!), twice.

Either choice is fine here — pick one and continue.

### What you get

```
Your identification has been saved in /Users/anna/.ssh/id_ed25519
Your public key has been saved in /Users/anna/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:xXxXxXxXxXxXxXxXxXxXxXxXxXxXx anna.example@gmail.com
The key's randomart image is:
+--[ED25519 256]--+
|      .o+o.      |
|     .  o=       |     ← a fun "fingerprint picture".
|      ...        |       Purely decorative — ignore it.
+----[SHA256]-----+
```

🎉 You now own a key pair. Look at it:

```
ls ~/.ssh
```

Expected (at least):

```
id_ed25519      id_ed25519.pub
```

There they are — 🗝️ private key (no ending) and 🔓 public key (`.pub`). One more time, because it is the only real rule of this notebook:

> ⚠️ **The file WITHOUT `.pub` never leaves your laptop.** Whenever anything — a website, a person, an instruction — asks for "your SSH key", it *always* means the **`.pub`** one.

### 4½ — Only if the key already existed

If Question 1 warned you that `id_ed25519 already exists` and you typed `n`: great, you simply already own a key pair — probably from an earlier setup. Everything in this notebook works with it as-is; just continue to section 5. (Can't remember its passphrase? Then, and only then, rerun `ssh-keygen` and answer `y` to overwrite — you'll get a fresh start, and just have to re-add the new public key wherever the old one was used.)

### Reading your public key (you'll need this twice today)

Print the **public** key to the screen — same command everywhere:

```
cat ~/.ssh/id_ed25519.pub
```

Expected — **one single long line**, like:

```
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIN7rZ1x…lots more…q8yF anna.example@gmail.com
```

That one line *is* your padlock 🔓. When you need to copy it: select the **whole line** with the mouse and copy it (Notebook 1's copy shortcuts) — or send it straight to the clipboard:

- 🍎 macOS: `pbcopy < ~/.ssh/id_ed25519.pub`
- 🪟 Windows (WSL): `clip.exe < ~/.ssh/id_ed25519.pub` — a neat trick: your Linux can call Windows tools, and `clip.exe` is the Windows clipboard
- 🐧 Linux: print it with `cat`, select the line, copy with **Ctrl+Shift+C**

> ✅ **Checkpoint** — You can print your public key and it starts with `ssh-ed25519` and ends with your email. You know which of the two files is shareable (the `.pub` one — the padlock).

## 5. Log into a remote server

> 📖 **Read-only for now — we do this section together in the lecture.** There is no course server running yet, so you cannot try this at home. Read it so every step is familiar; in class you will get your own username and server address, and we do it live, together.

Here is how it works. You need two pieces of information **from your instructors** (or, later in life, from whoever runs the server):

- a **username** on the server — say `anna42`
- the **server address** — say `digimaster.university.edu`

*(The boxes below use these example values — in the lecture you replace them with yours.)*

The command follows the pattern `ssh username@address`:

```
ssh anna42@digimaster.university.edu
```

### First connection — the fingerprint question

The very first time you connect to any server, SSH stops and asks:

```
The authenticity of host 'digimaster.university.edu (130.83.xx.xx)' can't be established.
ED25519 key fingerprint is SHA256:kQx9v…
Are you sure you want to continue connecting (yes/no/[fingerprint])?
```

Don't panic — this is **normal and correct**. SSH has never seen this server before and asks: *"should I trust it?"* (Paranoid organizations verify that fingerprint through a second channel; for the course server, it's fine to trust it.)

Type `yes` and press Enter. SSH remembers the server from now on (in `~/.ssh/known_hosts`) and will never ask again for this machine.

### The password

Next, the server asks for your **server password** (the one your instructors gave you — not your laptop's password!):

```
anna42@digimaster.university.edu's password:
```

Type it. **The screen shows nothing while you type — not even stars.** That is a security feature, not a malfunction. Type blind, press Enter.

### You are in! 🎉

```
Welcome to Ubuntu 24.04 LTS (GNU/Linux ...)
Last login: Mon Sep 7 09:12:44 2026 from 46.223.xx.xx
anna42@digimaster:~$
```

Look at that last line: **the prompt changed.** It now says `anna42@digimaster` — you are typing *on the server*. Try your Notebook-1 vocabulary; it all works, because the server is a Linux machine:

```
pwd
```
→ `/home/anna42` — your home folder **on the server** (not your laptop!)

```
ls
```
→ whatever exists there (maybe nothing yet — it's a fresh account)

```
hostname
```
→ the server's name. Proof of where you are.

> 💡 **One window, two computers.** This trips everyone up at first. *The prompt tells you where you are:* `anna@Annas-MacBook ~ %` = your laptop; `anna42@digimaster:~$` = the server. Glance at the prompt before every command until it becomes reflex.

### Coming home

```
exit
```

Expected: `Connection to digimaster.university.edu closed.` — and your prompt is your laptop's again. That's the whole loop: `ssh` in, work, `exit` out.

## 6. Hang your padlock on the server — passwordless login

> 📖 **Read-only for now — we do this section together in the lecture.** There is no course server running yet, so you cannot try this at home. Read it so every step is familiar; in class you will get your own username and server address, and we do it live, together.

Right now the server asks for your password every single time. Let's fix that properly: put your **public key** on the server, once.

One purpose-built command — same on every system (with **your** username & address):

```
ssh-copy-id anna42@digimaster.university.edu
```

It asks for the server password **one last time**, then answers:

```
Number of key(s) added: 1
Now try logging into the machine, with: "ssh 'anna42@digimaster.university.edu'"
```

### Now test it

```
ssh anna42@digimaster.university.edu
```

**No password question** — straight to the server prompt. (If you set a key passphrase in section 4, *that* may be asked instead — that question comes from your own laptop, guarding your key file.)

What actually happened: your padlock was added to a file on the server called `~/.ssh/authorized_keys` — the server's list of trusted padlocks. Anyone whose private key matches an entry gets in without a password. Log in and `cat ~/.ssh/authorized_keys` — you'll recognize your public key line.

Then `exit` again.

> ✅ **Checkpoint (in the lecture)** — once we have done this together, `ssh` into the course server works **without** the server password. What to take away *now* is simply what `ssh-copy-id` does: it adds your public key to the server's `~/.ssh/authorized_keys` list.

## 7. Nicknames for servers — the config file

> 📖 **Read-only for now — we do this section together in the lecture.** There is no course server running yet, so you cannot try this at home. Read it so every step is familiar; in class you will get your own username and server address, and we do it live, together.

Typing `anna42@digimaster.university.edu` forever gets old. SSH has an address book: a plain-text file at `~/.ssh/config`. Let's create it.

Open it in the terminal's little text editor `nano` (from Notebook 1's "nice to know" table — same on every system):

```
nano ~/.ssh/config
```

Type or paste this inside (with your values). Then save & exit: **Ctrl+O**, Enter, **Ctrl+X**.

```
Host uni
    HostName digimaster.university.edu
    User anna42
    IdentityFile ~/.ssh/id_ed25519
```

Reading it: *"When I say `uni`, I mean: that address, that username, that key."* The indented lines start with 4 spaces. From now on, your whole login is:

```
ssh uni
```

Add more servers as more `Host` blocks in the same file — one address book for your whole career.

## 8. Connect your key to GitHub — unlock `git push` 🔓

In Notebook 2 you could `clone` and `pull`, but not `push` — GitHub had no way to know it's really you. GitHub speaks SSH: hang your padlock there too, and it will.

### Step 1 — copy your public key

From section 4: `pbcopy < ~/.ssh/id_ed25519.pub` (🍎) · `clip.exe < ~/.ssh/id_ed25519.pub` (🪟 WSL) · print with `cat` & copy the line (🐧).

### Step 2 — paste it into GitHub

1. Log into <https://github.com>.
2. Click your **profile picture** (top-right) → **Settings**.
3. In the left menu: **SSH and GPG keys** → green button **New SSH key**.
4. **Title**: name the computer this key lives on, e.g. `Annas MacBook` (you'll add one key per machine over the years).
5. **Key**: paste — it must be the single `ssh-ed25519 AAAA…` line.
6. **Add SSH key** (GitHub may ask your GitHub password to confirm).

### Step 3 — test the handshake

```
ssh -T git@github.com
```

First time: the fingerprint question again (`yes` — same ritual as with the course server, this time it's GitHub's server). Expected answer:

```
Hi anna-example! You've successfully authenticated, but GitHub does not provide shell access.
```

> 💡 Read that carefully — it **looks** like an error but is the **success message**! ("You're authenticated" — GitHub knows who you are. "No shell access" — GitHub is not a server you log *into*; it only speaks Git.) If it greets you by your username, you have won.

### What changes now

GitHub has two address styles for every repository:

| Style | Looks like | Identity |
|---|---|---|
| HTTPS | `https://github.com/sid027/premaster-digimaster26.git` | none — read public repos only |
| **SSH** | `git@github.com:sid027/premaster-digimaster26.git` | **your key** — push allowed (where you have rights) |

From today, when you clone a repo you intend to *work* on, choose the **SSH address** (green **Code** button on GitHub → **SSH** tab). On your own repositories, `git push` will now simply work — Notebook 2's missing piece, delivered. 🤝

## 9. Bonus: moving files with `scp`

> 📖 **Read-only for now — we do this section together in the lecture.** There is no course server running yet, so you cannot try this at home. Read it so every step is familiar; in class you will get your own username and server address, and we do it live, together.

Once you can *log into* the server — but how do you get files there and back? The SSH family includes **`scp`** (**s**ecure **c**o**p**y). It behaves exactly like Notebook 1's `cp` — `scp source destination` — except one side can be remote. Remote sides are written `name@server:path` (with your `config` from section 7, `uni:path` is enough):

**Laptop → server** (upload `report.txt` to your server home folder):

```
scp report.txt uni:~/
```

**Server → laptop** (download `results.csv` to *here* — remember from Notebook 1: `.` means "the folder I'm standing in"):

```
scp uni:~/results.csv .
```

A whole folder: add `-r`, like with `rm`:

```
scp -r project_folder uni:~/
```

Each transfer prints a small progress line — and the files are encrypted on the way, of course. That's all you need for the course.

## 10. When something goes wrong

| Message / situation | What it usually means | Fix |
|---|---|---|
| `Permission denied (publickey,password)` after typing password | Wrong password or wrong **username** for this server | Retype carefully (it's invisible!), check the username your instructors gave you |
| `Permission denied (publickey)` — not even asked for a password | The server only accepts keys, and yours isn't on it yet | Do section 6 (`ssh-copy-id`); double-check username |
| `Connection timed out` / `Connection refused` | Address wrong or unreachable — often a network issue | Check the address; many university servers require **campus network or VPN** — are you on it? |
| `Could not resolve hostname` | Typo in the server address | Check character by character |
| Big scary block: `WARNING: REMOTE HOST IDENTIFICATION HAS CHANGED!` | The server's fingerprint differs from what SSH remembered (server reinstalled — or, in theory, tampering) | Don't just bulldoze past it: ask the instructors; if they confirm the server changed, run `ssh-keygen -R digimaster.university.edu` and connect again |
| `WARNING: UNPROTECTED PRIVATE KEY FILE` / `Bad permissions` | Your key file is readable by others; SSH refuses in protest | `chmod 700 ~/.ssh` then `chmod 600 ~/.ssh/id_ed25519` (`chmod` = the permissions tool; these two lines are the standard fix) |
| `ssh: command not found` (🪟) | You are in a Windows window, not your Linux | Open the **Ubuntu** terminal (Start → `Ubuntu`) and try again |
| GitHub test says `Permission denied (publickey)` | GitHub doesn't have your (correct) public key | Redo section 8 — was the **whole** `.pub` line pasted? |
| Asked for a passphrase you don't remember | Your key has a passphrase (section 4) you've forgotten | Make a fresh key (`ssh-keygen`, overwrite `y`) and re-add the new `.pub` to server & GitHub |

## 11. Cheat sheet 📋

```
KEYS                                        SERVERS
  ssh-keygen -t ed25519 -C "mail"             ssh user@server        log in
      make key pair (once per laptop)         exit                   log out
  ~/.ssh/id_ed25519      PRIVATE = stays!     ssh-copy-id user@srv   put padlock on server
  ~/.ssh/id_ed25519.pub  public  = share      ssh uni                log in via nickname
  cat ~/.ssh/id_ed25519.pub   show padlock
                                            FILES  (like cp, but remote)
GITHUB                                        scp file uni:~/        upload
  public key → github.com → Settings          scp uni:~/file .       download
      → SSH and GPG keys → New SSH key        scp -r folder uni:~/   whole folder
  ssh -T git@github.com     test ("Hi …!")
  clone with git@github.com:user/repo.git   ADDRESS BOOK  ~/.ssh/config
      → push works                            Host uni
                                                  HostName server.university.edu
REMEMBER                                          User anna42
  "your SSH key" always means the .pub one        IdentityFile ~/.ssh/id_ed25519
  invisible password typing is normal
  the prompt tells you which computer you're on
```

## 12. Check yourself ✅

**On your own, after this notebook:**

1. Can you explain the padlock/key picture — which file is which, and which one is secret?
2. Can you create a key pair, and find the two files afterwards?
3. Why does the first connection to a new server ask the fingerprint question — and is that bad?
4. What does the GitHub message *"Hi anna! You've successfully authenticated…"* mean?

**After we have done sections 5–7 and 9 together in the lecture:**

5. Can you log into a server, prove which computer you're on, and come back?
6. Can you make a server stop asking for your password? What file on the server did that change?
7. How would you copy a file from the server to your laptop?

---

**🎉 Done!** You have a professional digital identity: one key, and you already know how padlocks get hung — GitHub today, the course server in the lecture.

**➡️ Next: `04-docker`**, the final boss: run any software, anywhere, in standardized containers.